# Reverse-mode autodiff by hand

Autograd is bookkeeping. Run the function forward while recording every intermediate value,
then walk the record backward, multiplying by one local derivative per step and adding up the
contributions that reach the same variable. Here you keep the books yourself, in NumPy, for

$$
f(x, y) = (xy + x)\,\sigma(y),
$$

where $\sigma$ is the logistic sigmoid $1 / (1 + e^{-z})$, at $x = 1.5$ and $y = 0.5$,
with four recorded steps: $p = xy$, then $q = p + x$, then
$s = \sigma(y)$, and finally $f = qs$. Note that $x$ reaches $f$ by two routes, through $p$
and directly in $q$, and so does $y$, through $p$ and through $s$.

**How to work through it.** Each part asks for one function. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your function is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first.

In [ ]:
import numpy as np


def sigmoid(z):
    return 1 / (1 + np.exp(-z))


def f(x, y):
    p = x * y
    return (p + x) * sigmoid(y)


x0, y0 = 1.5, 0.5
print(f"f({x0}, {y0}) = {f(x0, y0):.6f}")

## Part 1 (coding task): The gradient on paper, then as a formula

Walk the four steps backward on paper. Start from $\bar f = 1$. At each step, for every
input $u$ of that step with output $v$, add $\bar v \cdot \partial v / \partial u$ to the
adjoint $\bar u$, adding rather than overwriting when a variable collects a second
contribution. The end result is a closed-form expression for each partial derivative in
terms of $x$, $y$, $q$ and $s$.

Write `gradient_formula(x, y)`, which returns the pair `(df_dx, df_dy)` from those
expressions, with no finite differences and no recorded steps.

In [ ]:
def gradient_formula(x, y):
    """(df/dx, df/dy) at (x, y), from the expressions you derived."""
    # TODO
    raise NotImplementedError

In [ ]:
def central_differences(x, y, h=1e-6):
    dx = (f(x + h, y) - f(x - h, y)) / (2 * h)
    dy = (f(x, y + h) - f(x, y - h)) / (2 * h)
    return dx, dy


dx, dy = gradient_formula(x0, y0)
fd_dx, fd_dy = central_differences(x0, y0)
print(f"formula: df/dx = {dx:.6f}  df/dy = {dy:.6f}")
print(f"central differences: df/dx = {fd_dx:.6f}  df/dy = {fd_dy:.6f}")
s0 = sigmoid(y0)
assert not np.isclose(dx, s0 * y0, atol=1e-6), "df/dx is missing the direct route of x into q = p + x: x collects one share through p and another through q"
assert np.isclose(dx, fd_dx, atol=1e-5), f"df/dx = {dx:.6f} disagrees with the central difference {fd_dx:.6f}"
assert not np.isclose(dy, s0 * x0, atol=1e-6), "df/dy is missing the route through s = sigmoid(y), whose slope is s * (1 - s)"
assert np.isclose(dy, fd_dy, atol=1e-5), f"df/dy = {dy:.6f} disagrees with the central difference {fd_dy:.6f}"
for x, y in [(0.3, -1.2), (-2.0, 0.8), (1.0, 3.0)]:
    gx, gy = gradient_formula(x, y)
    ex, ey = central_differences(x, y)
    assert np.isclose(gx, ex, atol=1e-5) and np.isclose(gy, ey, atol=1e-5), f"at ({x}, {y}) the formula gives ({gx:.5f}, {gy:.5f}) and central differences give ({ex:.5f}, {ey:.5f})"
print("Part 1 passed")

## Part 2 (coding task): The same walk as a tape

Write `reverse_mode(x, y)`, which returns `(value, df_dx, df_dy)`. Its forward pass computes
`p`, `q`, `s` and the output in that order and keeps all four. Its reverse pass starts from
an adjoint of `1.0` for the output and visits the steps in reverse order, `out = q * s`,
then `s = sigmoid(y)`, `q = p + x` and `p = x * y`, giving each step's inputs their share
and using `+=` for a variable that already has one. Use only the stored values and the local
derivative of each step. The closed forms from Part 1 stay out of it, and so do finite
differences.

In [ ]:
def reverse_mode(x, y):
    """(f, df/dx, df/dy): a forward pass that keeps every step, then a reverse pass over them."""
    # forward pass, every step kept
    p = x * y
    q = p + x
    s = sigmoid(y)
    out = q * s
    # reverse pass
    # TODO
    raise NotImplementedError

In [ ]:
value, dx, dy = reverse_mode(x0, y0)
print(f"tape: f = {value:.6f}  df/dx = {dx:.6f}  df/dy = {dy:.6f}")
assert np.isclose(value, f(x0, y0)), f"the forward pass gives {value:.6f}, f gives {f(x0, y0):.6f}"
assert np.isclose(dx, 0.933689, atol=1e-5), f"df/dx = {dx:.6f}, expected 0.933689: x gets a share from q = p + x and another from p = x * y, so add them"
assert np.isclose(dy, 1.462447, atol=1e-5), f"df/dy = {dy:.6f}, expected 1.462447: y gets a share from p = x * y and another from s = sigmoid(y)"
rng = np.random.default_rng(0)
for x, y in rng.normal(size=(5, 2)):
    _, gx, gy = reverse_mode(x, y)
    ex, ey = gradient_formula(x, y)
    assert np.isclose(gx, ex, atol=1e-9) and np.isclose(gy, ey, atol=1e-9), f"at ({x:.3f}, {y:.3f}) the tape gives ({gx:.6f}, {gy:.6f}) and Part 1's formula gives ({ex:.6f}, {ey:.6f})"
print("Part 2 passed")

## Solution

### Part 1: The gradient on paper, then as a formula

In [ ]:
def gradient_formula(x, y):
    """(df/dx, df/dy) at (x, y), from the expressions you derived."""
    s = sigmoid(y)
    q = x * y + x
    df_dx = s * (y + 1)
    df_dy = s * x + q * s * (1 - s)
    return df_dx, df_dy

**Why it works.** Read the four steps from the last to the first. $f = qs$ hands $\bar q = s$
and $\bar s = q$. $s = \sigma(y)$ hands $y$ the share $\bar s\,\sigma'(y) = q\,s(1 - s)$.
$q = p + x$ has local derivative 1 toward both inputs, so $\bar p = s$ and $x$ gets a first
share of $s$. $p = xy$ hands $x$ a second share, $\bar p\,y = sy$, and hands $y$ a second
share, $\bar p\,x = sx$. Adding the shares,

$$
\begin{aligned}
\frac{\partial f}{\partial x} &= s(1 + y), \\
\frac{\partial f}{\partial y} &= sx + qs(1 - s).
\end{aligned}
$$

At $(1.5, 0.5)$, $s = 0.6225$ and $q = 2.25$, so the two derivatives are $0.9337$ and
$1.4624$. Dropping the direct route of $x$ leaves $sy = 0.311$, a third of the right value,
and that is the share most people forget.

### Part 2: The same walk as a tape

In [ ]:
def reverse_mode(x, y):
    """(f, df/dx, df/dy): a forward pass that keeps every step, then a reverse pass over them."""
    # forward pass, every step kept
    p = x * y
    q = p + x
    s = sigmoid(y)
    out = q * s
    # reverse pass
    d_out = 1.0
    dq = d_out * s             # out = q * s
    ds = d_out * q
    dy = ds * s * (1 - s)      # s = sigmoid(y)
    dp = dq                    # q = p + x
    dx = dq
    dx += dp * y               # p = x * y, the second share for x
    dy += dp * x
    return out, dx, dy

**Why it works.** The tape is the four lines of the forward pass, read bottom to top. Each line
passes the adjoint of its output, times the local derivative, to its inputs: `out = q * s`
gives `s` to `q` and `q` to `s`, `s = sigmoid(y)` gives `ds * s * (1 - s)` to `y`,
`q = p + x` gives `dq` unchanged to both `p` and `x`, and `p = x * y` gives `dp * y` to `x`
and `dp * x` to `y`. A variable that was used twice collects twice, which is what the two
`+=` lines do, and the result agrees with Part 1 to the last digit. `loss.backward()` runs
the same walk over a graph of thousands of nodes, and the local derivative is the only
thing each operation has to know about itself.